# 02 · Spike sorting with SpyKING CIRCUS 2
Groups detected spikes into units (putative neurons). Runs on CPU (no GPU needed).

## 1. Setup
Load the MaxOne test file. `HDF5_PLUGIN_PATH` must be set before importing SpikeInterface so HDF5 can read MaxWell's compressed data.

In [4]:
import os # access environment settings
from pathlib import Path # file paths
os.environ["HDF5_PLUGIN_PATH"] = str(Path.home() / "hdf5_plugin_path_maxwell") # tell HDF5 where MaxWell's decompression plugin is; must run before importing spikeinterface

import numpy as np # arrays and maths
import matplotlib.pyplot as plt # plotting
import spikeinterface as si # core SpikeInterface
import spikeinterface.extractors as se # file readers
import spikeinterface.preprocessing as spre # cleaning tools
import spikeinterface.sorters as ss # spike sorters

file_path = Path("../data/raw/maxone_record_000011.raw.h5") # the test file
rec = se.read_maxwell(file_path) # open the recording
rec_s = spre.unsigned_to_signed(rec) # uint16 -> signed, ready for the sorter
dur = rec.get_total_duration() # length in s, for firing rates
print(rec) # check it loaded
from pathlib import Path # file paths
import numpy as np # arrays and maths
import matplotlib.pyplot as plt # plotting
import spikeinterface as si # core SpikeInterface
import spikeinterface.extractors as se # file readers
import spikeinterface.preprocessing as spre # cleaning tools
import spikeinterface.sorters as ss # spike sorters

file_path = Path("../data/raw/maxone_record_000011.raw.h5") # the test file
rec = se.read_maxwell(file_path) # open the recording
rec_s = spre.unsigned_to_signed(rec) # uint16 -> signed, ready for the sorter
dur = rec.get_total_duration() # length in s, for firing rates
print(rec) # check it loaded

The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!
MaxwellRecordingExtractor: 25 channels - 20.0kHz - 1 segments - 1,201,200 samples 
                           60.06s (1.00 minutes) - uint16 dtype - 57.28 MiB
  file_path: /Users/arelibalderas/git_repos/hdmea-organoid-pipeline/notebooks/../data/raw/maxone_record_000011.raw.h5
The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!
MaxwellRecordingExtractor: 25 channels - 20.0kHz - 1 segments - 1,201,200 samples 
                           60.06s (1.00 minutes) - uint16 dtype - 57.28 MiB
  file_path: /Users/arelibalderas/git_repos/hdmea-organoid-pipeline/notebooks/../data/raw/maxone_record_000011.raw.h5


## 2. Sort, save and reload
SC2 does its own filtering, detection, clustering and template matching.
Settings: `deterministic_peaks_detection=True`; motion correction off (70 µm patch is too small to estimate drift).
Sorting is not fully reproducible: core units are stable, borderline units vary between runs. So we sort once (`run_sorting = True`), save, and reload (`run_sorting = False`).
Units and spike counts: see output below. Compare with 01: 4404 electrode-level detections.

In [5]:
si.set_global_job_kwargs(n_jobs=4, progress_bar=True) # back to 4 cores (1 core didn't fix it)
sort_dir = Path("../data/sorting/sc2_test") # where the sorting is saved (inside data/, ignored by Git)

run_sorting = False # True = run a NEW sorting and overwrite the saved one; False = load the saved one
if run_sorting: # only when you ask for it
    sorting = ss.run_sorter(
        "spykingcircus2", # which sorter
        rec_s, # signed, unfiltered recording: SC2 filters it itself
        folder=sort_dir, # where results are written
        remove_existing_folder=True, # overwrite old results
        verbose=True, # print progress
        deterministic_peaks_detection=True, # detect all peaks, no early stopping
        apply_motion_correction=False, # no drift correction on a 70 µm patch
    )
else:
    sorting = ss.read_sorter_folder(sort_dir) # load the saved result: same units every time
print(sorting) # summary: number of units, duration

units = sorting.get_unit_ids() # one ID per neuron ("unit")
print(f"\nUnits found: {len(units)}") # number of neurons
n_total = 0 # running total of spikes
for u in units: # one row per unit
    n = len(sorting.get_unit_spike_train(u)) # spikes of this unit
    n_total += n # add to total
    print(f"Unit {u}: {n} spikes, {n / dur:.2f} Hz") # count and firing rate
print(f"Total sorted spikes: {n_total}") # compare with 01: 4404 electrode-level detections

The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!
NumpyFolder (NumpyFolderSorting): 12 units - 1 segments - 20.0kHz

Units found: 12
Unit 1: 65 spikes, 1.08 Hz
Unit 3: 94 spikes, 1.57 Hz
Unit 5: 65 spikes, 1.08 Hz
Unit 7: 31 spikes, 0.52 Hz
Unit 12: 56 spikes, 0.93 Hz
Unit 15: 73 spikes, 1.22 Hz
Unit 17: 46 spikes, 0.77 Hz
Unit 18: 127 spikes, 2.11 Hz
Unit 19: 99 spikes, 1.65 Hz
Unit 21: 64 spikes, 1.07 Hz
Unit 23: 60 spikes, 1.00 Hz
Unit 24: 135 spikes, 2.25 Hz
Total sorted spikes: 915


## 3. Quality check
SNR = average spike amplitude ÷ noise σ (≥ 5 = clear unit).
Refractory violations = % of ISIs < 1.5 ms (< 1% = single neuron; more = mixed units).

In [9]:
from spikeinterface.curation import remove_duplicated_spikes # curation tool for double-counted spikes

fs = rec.get_sampling_frequency() # samples per second (20000 Hz)
dup_ms = 0.3 # spikes closer than this in the same unit = duplicates; change freely
sorting_clean = remove_duplicated_spikes(sorting, censored_period_ms=dup_ms, method="keep_first") # keep the first, drop the copy
print("Spikes before:", sum(len(sorting.get_unit_spike_train(u)) for u in sorting.get_unit_ids())) # total before cleaning
print("Spikes after: ", sum(len(sorting_clean.get_unit_spike_train(u)) for u in sorting_clean.get_unit_ids())) # total after cleaning

rec_f = spre.bandpass_filter(rec_s, freq_min=300, freq_max=6000, dtype="float32") # filtered copy for spike shapes
analyzer = si.create_sorting_analyzer(sorting_clean, rec_f, sparse=True, format="memory") # links cleaned spikes to voltage
analyzer.compute(["random_spikes", "waveforms", "templates", "noise_levels"]) # shapes, average template, noise
qm = analyzer.compute("quality_metrics", metric_names=["snr"]).get_data() # SNR per unit

isi_limit_ms = 1.5 # refractory period in ms; change freely
snr_min = 5 # minimum SNR; change freely
viol_max = 1.0 # maximum % violations for a single neuron; change freely

print(f"\n{'unit':>4} {'spikes':>6} {'rate Hz':>7} {'SNR':>5} {'ISI<1.5ms':>9} {'viol %':>6}  label") # header
labels = {} # unit ID -> "single" or "MUA"
for u in sorting_clean.get_unit_ids(): # one row per unit
    st = sorting_clean.get_unit_spike_train(u) # spike times in samples
    isi_ms = np.diff(st) / fs * 1000 # ISIs in ms
    n_viol = int((isi_ms < isi_limit_ms).sum()) # ISIs shorter than refractory period
    pct = 100 * n_viol / max(len(isi_ms), 1) # % of ISIs
    snr = qm.loc[u, "snr"] # SNR of unit u
    labels[u] = "single" if (snr >= snr_min and pct <= viol_max) else "MUA" # MUA = multi-unit activity
    print(f"{u:>4} {len(st):>6} {len(st) / dur:>7.2f} {snr:>5.1f} {n_viol:>9} {pct:>6.1f}  {labels[u]}") # row
n_single = sum(v == "single" for v in labels.values()) # count single units
print(f"\nSingle units: {n_single}, MUA: {len(labels) - n_single}") # summary

Spikes before: 915
Spikes after:  901


compute_waveforms (workers: 4 processes spawn):   0%|          | 0/61 [00:00<?, ?it/s]

The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!
The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!
The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!
The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!


noise_level (workers: 4 processes spawn):   0%|          | 0/20 [00:00<?, ?it/s]

The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!
The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!
The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!
The h5 compression library for Maxwell is already located in /Users/arelibalderas/hdf5_plugin_path_maxwell/libcompression.dylib!

unit spikes rate Hz   SNR ISI<1.5ms viol %  label
   1     64    1.07   7.6         5    7.9  MUA
   3     89    1.48   9.2         4    4.5  MUA
   5     64    1.07   8.4         2    3.2  MUA
   7     31    0.52  11.9         0    0.0  single
  12     55    0.92   6.1         1    1.9  MUA
  15     73    1.22  19.3         3    4.2  MUA
  17     44    0.73   6.8         1    2.3  MUA
  18    126    2.10   7.9        13   10.4  MUA
  19     97    1.62  10.1         9    9.4  MU

In [10]:
for u in sorting_clean.get_unit_ids(): # one line per unit
    st = sorting_clean.get_unit_spike_train(u) # spike times in samples
    isi_ms = np.diff(st) / fs * 1000 # ISIs in ms
    short = np.sort(isi_ms[isi_ms < isi_limit_ms]) # only the violating ISIs, smallest first
    print(f"Unit {u}: {np.round(short, 2)}") # e.g. [0.05 0.1 ...] = duplicates; [0.6 1.2 ...] = mixed neurons


Unit 1: [0.35 0.7  1.4  1.4  1.4 ]
Unit 3: [0.35 0.6  1.1  1.15]
Unit 5: [0.95 1.05]
Unit 7: []
Unit 12: [0.55]
Unit 15: [0.4  0.6  0.85]
Unit 17: [0.4]
Unit 18: [0.45 0.45 0.5  0.55 0.55 0.55 0.75 0.8  1.15 1.25 1.25 1.3  1.35]
Unit 19: [0.35 0.7  1.   1.   1.05 1.05 1.25 1.35 1.35]
Unit 21: [0.65 0.75 0.95 1.  ]
Unit 23: [0.6  0.6  0.85 0.9  1.05]
Unit 24: [0.75 0.8  1.15 1.25 1.3  1.3 ]


**Result:** after removing duplicates (≤ 0.3 ms), most units still show refractory violations at 0.4–1.4 ms, i.e. mixed neurons, so they're labelled MUA. Expected for a 60 s, 25-channel test file: too few electrodes and spikes per unit for clean separation. Pipeline validated; real organoid recordings next.